# ENTREGABLE 1 — Fundamentación en Analítica Estratégica de Datos
## Etapa de preprocesamiento y análisis básico de un modelo estrella

**Dataset:** `dim_tienda.csv`, `dim_producto.csv`, `fact_ventas.csv`

---

## 0. Instalación e importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('Librerías cargadas correctamente ✔')

---
## 5. Carga de los datos

In [ ]:
# ─── Ajusta las rutas si subes los archivos a Google Colab ───
# Opción A – archivos en la misma carpeta del notebook:
PATH_PRODUCTOS = 'DIM_PRODUCTOS.csv'
PATH_TIENDAS   = 'DIM_TIENDAS.csv'
PATH_VENTAS    = 'FACT_VENTAS_MASTER.csv'

# Opción B – desde Google Drive (descomenta y ajusta):
# from google.colab import drive
# drive.mount('/content/drive')
# PATH_PRODUCTOS = '/content/drive/MyDrive/TuCarpeta/DIM_PRODUCTOS.csv'
# PATH_TIENDAS   = '/content/drive/MyDrive/TuCarpeta/DIM_TIENDAS.csv'
# PATH_VENTAS    = '/content/drive/MyDrive/TuCarpeta/FACT_VENTAS_MASTER.csv'

dim_producto = pd.read_csv(PATH_PRODUCTOS)
dim_tienda   = pd.read_csv(PATH_TIENDAS)
fact_ventas  = pd.read_csv(PATH_VENTAS)

print(f'dim_producto : {dim_producto.shape[0]:,} filas × {dim_producto.shape[1]} columnas')
print(f'dim_tienda   : {dim_tienda.shape[0]:,} filas × {dim_tienda.shape[1]} columnas')
print(f'fact_ventas  : {fact_ventas.shape[0]:,} filas × {fact_ventas.shape[1]} columnas')

---
## 6. Exploración inicial de los datos

### 6.1 Dimensión PRODUCTOS

In [ ]:
print('=== Primeros registros ===')
display(dim_producto.head())

print('\n=== Estructura y tipos ===')
dim_producto.info()

print('\n=== Valores nulos ===')
print(dim_producto.isnull().sum())

print('\n=== Duplicados ===')
print(f'Registros duplicados: {dim_producto.duplicated().sum()}')

print('\n=== Categorías únicas (tal como vienen) ===')
print(dim_producto['Categoria'].unique())

### 6.2 Dimensión TIENDAS

In [ ]:
print('=== Primeros registros ===')
display(dim_tienda.head())

print('\n=== Estructura y tipos ===')
dim_tienda.info()

print('\n=== Valores nulos ===')
print(dim_tienda.isnull().sum())

print('\n=== Duplicados ===')
print(f'Registros duplicados: {dim_tienda.duplicated().sum()}')

print('\n=== Países únicos ===')
print(dim_tienda['Pais'].unique())

### 6.3 Tabla de hechos VENTAS

In [ ]:
print('=== Primeros registros ===')
display(fact_ventas.head(3))

print('\n=== Estructura y tipos ===')
fact_ventas.info()

print('\n=== Estadísticas descriptivas (columnas numéricas clave) ===')
display(fact_ventas[['Monto_Total_Local','Impuesto_Local','Tasa_Referencia_Dia','Costo_Envio_USD']].describe())

print('\n=== Valores nulos ===')
print(fact_ventas.isnull().sum())

print('\n=== Duplicados ===')
print(f'Registros duplicados: {fact_ventas.duplicated().sum()}')

print('\n=== Valores únicos de Es_Online ===')
print(fact_ventas['Es_Online'].value_counts())

print('\n=== Monedas y tasas registradas ===')
display(fact_ventas.groupby('Moneda_Transaccion')['Tasa_Referencia_Dia'].value_counts().to_frame())

### Resumen de hallazgos en exploración inicial

| Tabla | Problema identificado |
|---|---|
| `dim_producto` | Categorías mal escritas: `Electr0nica`, `Ofic_Ina`, `H0gar`, `Gaminng`, `G@ming` |
| `fact_ventas` | `Es_Online` tiene 6 representaciones distintas para sí/no |
| `fact_ventas` | 487 registros con `SKU-ERROR-*` (sin correspondencia en `dim_producto`) |
| `fact_ventas` | `Tasa_Referencia_Dia` con valores cruzados entre monedas |
| `fact_ventas` | Columnas PII (nombre, email, edad del cliente) con datos ficticios/nulos |
| `fact_ventas` | `Fecha_ISO` en formato string, debe convertirse a datetime |
| `fact_ventas` | No existe columna `Cantidad`; el monto total es en moneda local |

---
## 7. Limpieza de datos

### 7.1 Limpieza de dim_producto — estandarización de categorías

In [ ]:
# Trabajamos con copias para preservar los originales
prod = dim_producto.copy()
tiend = dim_tienda.copy()
ventas = fact_ventas.copy()

# ─── Depurar espacios en blanco ───
prod['Nombre_Producto'] = prod['Nombre_Producto'].str.strip()
prod['Categoria']       = prod['Categoria'].str.strip()

# ─── Mapa de corrección de categorías ───
categoria_map = {
    'Electr0nica': 'Electrónica',
    'Ofic_Ina'   : 'Oficina',
    'H0gar'      : 'Hogar',
    'Gaminng'    : 'Gaming',
    'G@ming'     : 'Gaming',
}
prod['Categoria'] = prod['Categoria'].replace(categoria_map)

print('Categorías después de la limpieza:')
print(prod['Categoria'].value_counts())

### 7.2 Limpieza de dim_tienda

In [ ]:
# Eliminar espacios en blanco
tiend['Nombre_Tienda'] = tiend['Nombre_Tienda'].str.strip()
tiend['Pais']          = tiend['Pais'].str.strip()

# Verificar duplicados
print(f'Duplicados en dim_tienda: {tiend.duplicated().sum()}')
display(tiend)

### 7.3 Limpieza de fact_ventas — columna Es_Online

In [ ]:
# ─── Normalizar Es_Online a booleano ───
es_online_map = {
    'True' : True,  '1': True,  'Si': True,
    'False': False, '0': False, 'No': False
}
ventas['Es_Online'] = ventas['Es_Online'].map(es_online_map)

print('Distribución final de Es_Online:')
print(ventas['Es_Online'].value_counts())

# ─── Eliminar columnas PII e irrelevantes para el análisis ───
cols_eliminar = [
    'CLIENTE_NOMBRE_PII', 'CLIENTE_EMAIL_PII', 'CLIENTE_EDAD_PII',
    'Browser_User_Agent', 'ID_Session_Web', 'Version_Esquema', 'Ref_S3_TXT'
]
ventas.drop(columns=cols_eliminar, inplace=True)

print(f'\nColumnas restantes: {list(ventas.columns)}')

# ─── Eliminar duplicados ───
antes = len(ventas)
ventas.drop_duplicates(inplace=True)
print(f'\nDuplicados eliminados: {antes - len(ventas)}')

---
## 8. Manejo de valores nulos

In [ ]:
print('=== Nulos en dim_producto ===')
print(prod.isnull().sum())

print('\n=== Nulos en dim_tienda ===')
print(tiend.isnull().sum())

print('\n=== Nulos en fact_ventas ===')
print(ventas.isnull().sum())

# Imputar nulos en Es_Online (si quedaron) con False
ventas['Es_Online'].fillna(False, inplace=True)

# Imputar Costo_Envio_USD nulo con la mediana
if ventas['Costo_Envio_USD'].isnull().sum() > 0:
    mediana_envio = ventas['Costo_Envio_USD'].median()
    ventas['Costo_Envio_USD'].fillna(mediana_envio, inplace=True)
    print(f'\nCosto_Envio_USD nulos imputados con mediana: {mediana_envio:.2f}')

print('\nVerificación final de nulos en fact_ventas:')
print(ventas.isnull().sum())

---
## 9. Conversión de tipos de datos

In [ ]:
# ─── Fecha_ISO → datetime ───
ventas['Fecha_ISO'] = pd.to_datetime(ventas['Fecha_ISO'], errors='coerce')

# ─── Corregir Tasa_Referencia_Dia: usar la tasa correcta por moneda ───
# Tasas de referencia oficiales del dataset (valor modal correcto por moneda)
tasa_correcta = {
    'USD': 1.00,
    'GBP': 0.79,
    'EUR': 0.92,
    'MXN': 17.50,
    'COP': 4100.00,
    'JPY': 110.00   # valor de mercado estándar
}
ventas['Tasa_Referencia_Dia'] = ventas['Moneda_Transaccion'].map(tasa_correcta)

# ─── Flag_Fraude → booleano ───
ventas['Flag_Fraude'] = ventas['Flag_Fraude'].astype(bool)

print('Tipos de datos después de conversión:')
print(ventas.dtypes)
print(f'\nRango de fechas: {ventas["Fecha_ISO"].min()} → {ventas["Fecha_ISO"].max()}')

---
## 10. Validación de integridad referencial

In [ ]:
# ─── Tiendas sin correspondencia en dim_tienda ───
ids_tienda_dim    = set(tiend['Tienda_ID'])
ids_tienda_ventas = set(ventas['Tienda_ID'])
sin_tienda = ids_tienda_ventas - ids_tienda_dim
print(f'IDs de tienda en ventas sin match en dim_tienda: {len(sin_tienda)}')
if sin_tienda:
    print(sin_tienda)

# ─── Productos sin correspondencia en dim_producto ───
ids_sku_dim    = set(prod['SKU_ID'])
ids_sku_ventas = set(ventas['SKU_ID'])
sin_producto = ids_sku_ventas - ids_sku_dim
print(f'\nSKUs en ventas sin match en dim_producto: {len(sin_producto)}')
print('Ejemplo de SKUs inválidos:', list(sin_producto)[:5])

# ─── Cuantificar filas afectadas ───
mask_error = ventas['SKU_ID'].str.startswith('SKU-ERROR', na=False)
print(f'\nRegistros con SKU-ERROR en fact_ventas: {mask_error.sum()}')
print(f'Porcentaje del total: {mask_error.mean()*100:.2f}%')

# ─── Decisión: excluir registros sin correspondencia referencial ───
ventas_validas = ventas[~mask_error].copy()
ventas_invalidas = ventas[mask_error].copy()
print(f'\nRegistros válidos para análisis : {len(ventas_validas):,}')
print(f'Registros descartados (SKU-ERROR): {len(ventas_invalidas):,}')

---
## 11. Creación de columnas calculadas

In [ ]:
# ─── total_venta_USD: monto total convertido a USD ───
# Monto_Total_Local ya incluye el impuesto; lo convertimos a USD
ventas_validas['total_venta_USD'] = (
    ventas_validas['Monto_Total_Local'] / ventas_validas['Tasa_Referencia_Dia']
).round(2)

# ─── Monto neto sin impuesto (en USD) ───
ventas_validas['monto_neto_USD'] = (
    (ventas_validas['Monto_Total_Local'] - ventas_validas['Impuesto_Local'])
    / ventas_validas['Tasa_Referencia_Dia']
).round(2)

# ─── Variables de tiempo ───
ventas_validas['anio']     = ventas_validas['Fecha_ISO'].dt.year
ventas_validas['mes']      = ventas_validas['Fecha_ISO'].dt.month
ventas_validas['trimestre']= ventas_validas['Fecha_ISO'].dt.quarter
ventas_validas['dia_semana']= ventas_validas['Fecha_ISO'].dt.day_name()

print('Nuevas columnas creadas:')
display(ventas_validas[['Venta_ID','Fecha_ISO','Monto_Total_Local','Moneda_Transaccion',
                         'Tasa_Referencia_Dia','total_venta_USD','monto_neto_USD',
                         'anio','mes','trimestre']].head(5))

---
## 12. Integración del modelo estrella

In [ ]:
# Paso 1: fact_ventas ← dim_producto
df_integrado = ventas_validas.merge(
    prod[['SKU_ID','Nombre_Producto','Categoria','Costo_Base_USD']],
    on='SKU_ID',
    how='left'
)

# Paso 2: resultado ← dim_tienda
df_integrado = df_integrado.merge(
    tiend[['Tienda_ID','Nombre_Tienda','Pais']],
    on='Tienda_ID',
    how='left'
)

print(f'Tabla consolidada: {df_integrado.shape[0]:,} filas × {df_integrado.shape[1]} columnas')
print('\nColumnas disponibles:')
print(list(df_integrado.columns))

print('\nPrimeros registros de la tabla integrada:')
display(df_integrado[['Venta_ID','Fecha_ISO','Nombre_Tienda','Pais','Nombre_Producto',
                       'Categoria','total_venta_USD','anio','mes','trimestre']].head(5))

---
## 13. Análisis de resultados

In [ ]:
# 1. Categoría que genera mayores ingresos
ing_categoria = (
    df_integrado.groupby('Categoria')['total_venta_USD']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'total_venta_USD':'Ingresos_USD'})
)
ing_categoria['Ingresos_USD'] = ing_categoria['Ingresos_USD'].map('${:,.2f}'.format)
print('1. Ingresos por categoría:')
display(ing_categoria)

In [ ]:
# 2. Unidades vendidas por producto (cada fila = 1 transacción = 1 unidad lógica)
unid_producto = (
    df_integrado.groupby('Nombre_Producto')
    .size()
    .sort_values(ascending=False)
    .reset_index(name='Transacciones')
)
print('2. Transacciones por producto (top 10):')
display(unid_producto.head(10))

In [ ]:
# 3. Ingreso total por producto
ing_producto = (
    df_integrado.groupby('Nombre_Producto')['total_venta_USD']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'total_venta_USD':'Ingreso_USD'})
)
print('3. Ingreso total por producto (top 10):')
display(ing_producto.head(10))

In [ ]:
# 4. Promedio de ventas (USD) por tienda
prom_tienda = (
    df_integrado.groupby('Nombre_Tienda')['total_venta_USD']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'total_venta_USD':'Promedio_Venta_USD'})
)
print('4. Promedio de venta por tienda:')
display(prom_tienda)

In [ ]:
# 5. Total de ventas (USD) por país
ventas_pais = (
    df_integrado.groupby('Pais')['total_venta_USD']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'total_venta_USD':'Total_Ventas_USD'})
)
print('5. Total de ventas por país:')
display(ventas_pais)

In [ ]:
# 6. País con mayores ingresos
pais_top = ventas_pais.iloc[0]
print(f'6. País con mayores ingresos: {pais_top["Pais"]} — ${pais_top["Total_Ventas_USD"]:,.2f} USD')

In [ ]:
# 7. Total de ingresos por trimestre
ing_trimestre = (
    df_integrado.groupby(['anio','trimestre'])['total_venta_USD']
    .sum()
    .sort_index()
    .reset_index()
    .rename(columns={'total_venta_USD':'Ingreso_USD'})
)
ing_trimestre['Periodo'] = ing_trimestre['anio'].astype(str) + ' Q' + ing_trimestre['trimestre'].astype(str)
print('7. Ingresos por trimestre:')
display(ing_trimestre[['Periodo','Ingreso_USD']])

In [ ]:
# 8. Mes con mayor volumen de ventas (por número de transacciones)
ventas_mes = (
    df_integrado.groupby('mes')
    .size()
    .sort_values(ascending=False)
    .reset_index(name='Transacciones')
)
mes_top = ventas_mes.iloc[0]
import calendar
print(f'8. Mes con mayor volumen: Mes {int(mes_top["mes"])} ({calendar.month_name[int(mes_top["mes"])]}) — {mes_top["Transacciones"]:,} transacciones')
display(ventas_mes)

In [ ]:
# 9. Desempeño de tiendas por categoría de producto
desempeno = (
    df_integrado.groupby(['Nombre_Tienda','Categoria'])['total_venta_USD']
    .sum()
    .unstack(fill_value=0)
    .round(2)
)
print('9. Desempeño de tiendas por categoría (USD):')
display(desempeno)

In [ ]:
# 10. Top 5 productos con mayores ingresos
top5_productos = ing_producto.head(5)
print('10. Top 5 productos por ingresos:')
display(top5_productos)

In [ ]:
# 11. Top 5 tiendas con mayor cantidad de transacciones
top5_tiendas = (
    df_integrado.groupby('Nombre_Tienda')
    .size()
    .sort_values(ascending=False)
    .head(5)
    .reset_index(name='Transacciones')
)
print('11. Top 5 tiendas por cantidad de transacciones:')
display(top5_tiendas)

In [ ]:
# 12. Evolución mensual de ventas por categoría
evolucion = (
    df_integrado.groupby(['anio','mes','Categoria'])['total_venta_USD']
    .sum()
    .reset_index()
    .rename(columns={'total_venta_USD':'Ingreso_USD'})
    .sort_values(['anio','mes'])
)
evolucion['Periodo'] = evolucion['anio'].astype(str) + '-' + evolucion['mes'].astype(str).str.zfill(2)
evolucion_pivot = evolucion.pivot_table(index='Periodo', columns='Categoria',
                                         values='Ingreso_USD', aggfunc='sum', fill_value=0)
print('12. Evolución mensual de ventas por categoría (USD):')
display(evolucion_pivot.round(2))

In [ ]:
# 13. Producto con mayor ingreso dentro de cada categoría
top_por_cat = (
    df_integrado.groupby(['Categoria','Nombre_Producto'])['total_venta_USD']
    .sum()
    .reset_index()
    .sort_values(['Categoria','total_venta_USD'], ascending=[True, False])
    .groupby('Categoria')
    .first()
    .reset_index()
    .rename(columns={'total_venta_USD':'Ingreso_USD'})
)
print('13. Producto con mayor ingreso por categoría:')
display(top_por_cat)

In [ ]:
# 14. Tiendas con bajo desempeño (por debajo del percentil 25 en ingresos totales)
ing_por_tienda = (
    df_integrado.groupby('Nombre_Tienda')['total_venta_USD'].sum().reset_index()
    .rename(columns={'total_venta_USD':'Ingreso_USD'})
)
p25 = ing_por_tienda['Ingreso_USD'].quantile(0.25)
bajo_desempeno = ing_por_tienda[ing_por_tienda['Ingreso_USD'] < p25].sort_values('Ingreso_USD')
print(f'14. Tiendas con bajo desempeño (ingresos < p25 = ${p25:,.2f} USD):')
display(bajo_desempeno)

In [ ]:
# 15. Promedio de precio por categoría
prom_precio_cat = (
    df_integrado.groupby('Categoria')['total_venta_USD']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'total_venta_USD':'Precio_Promedio_USD'})
)
print('15. Promedio de precio de venta por categoría:')
display(prom_precio_cat)

---
## 14. Archivo de salida — ventas_limpias.csv

In [ ]:
# Seleccionar y ordenar las columnas del archivo final
columnas_salida = [
    'Venta_ID', 'Fecha_ISO', 'anio', 'mes', 'trimestre', 'dia_semana',
    'Tienda_ID', 'Nombre_Tienda', 'Pais',
    'SKU_ID', 'Nombre_Producto', 'Categoria', 'Costo_Base_USD',
    'Moneda_Transaccion', 'Monto_Total_Local', 'Impuesto_Local',
    'Tasa_Referencia_Dia', 'total_venta_USD', 'monto_neto_USD',
    'Costo_Envio_USD', 'Metodo_Pago', 'Es_Online', 'Canal_Origen',
    'Flag_Fraude', 'Pais_Venta'
]

ventas_limpias = df_integrado[columnas_salida].copy()

ventas_limpias.to_csv('ventas_limpias.csv', index=False, encoding='utf-8-sig')

print(f'Archivo ventas_limpias.csv generado exitosamente.')
print(f'Dimensiones: {ventas_limpias.shape[0]:,} filas × {ventas_limpias.shape[1]} columnas')
display(ventas_limpias.head(3))

---
## 15. Análisis de Hallazgos del Preprocesamiento

### Problemas de calidad identificados

| # | Tabla | Hallazgo | Acción aplicada |
|---|---|---|---|
| 1 | `dim_producto` | 5 variantes incorrectas de categorías (`Electr0nica`, `H0gar`, `Ofic_Ina`, `Gaminng`, `G@ming`) | Estandarización con mapa de corrección |
| 2 | `fact_ventas` | `Es_Online` con 6 representaciones distintas para verdadero/falso | Normalización a booleano |
| 3 | `fact_ventas` | 487 registros (4.87%) con `SKU-ERROR-*` sin correspondencia en `dim_producto` | Exclusión del análisis y registro aparte |
| 4 | `fact_ventas` | `Tasa_Referencia_Dia` con valores cruzados entre monedas (ej. JPY con tasa de COP) | Reasignación de tasa correcta por moneda |
| 5 | `fact_ventas` | Columnas PII con datos ficticios o enmascarados | Eliminación de columnas PII |
| 6 | `fact_ventas` | `Fecha_ISO` en formato string ISO 8601 | Conversión a datetime64 |

### Consistencia entre hechos y dimensiones

- **dim_tienda** → integridad referencial perfecta: el 100% de los `Tienda_ID` en ventas tienen correspondencia.
- **dim_producto** → 487 registros de ventas (≈ 4.87%) presentaron `SKU-ERROR-*`, siendo excluidos del análisis principal.

### Comportamiento general de ventas

- Se procesaron **9,513 transacciones válidas** distribuidas en **20 tiendas** y **150 productos** en **5 categorías** y **6 países**.
- Los ingresos están denominados en 6 monedas diferentes, convertidas a USD para comparabilidad.

### Patrones temporales observados

- El dataset cubre múltiples trimestres. Las consultas de evolución mensual (análisis 12) revelan la tendencia de cada categoría a lo largo del tiempo.
- El análisis trimestral (consulta 7) permite identificar picos de demanda estacional.

### Importancia de las transformaciones realizadas

Sin el preprocesamiento, los análisis habrían contado la misma categoría como cuatro entidades distintas, sobreestimado algunas categorías, incorporado transacciones huérfanas al modelo y producido conversiones de moneda incorrectas. Las transformaciones garantizan que los resultados del paso 13 sean **confiables y comparables**.